In [24]:
import pandas as pd
import networkx as nx
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# CONFIGURATION
# ==========================================
YEARS = [2018, 2019, 2020, 2021, 2022, 2023]
SIMILARITY_THRESHOLD = 0.90  # Adjusted for whitelisted data

FOOD_WHITELIST = [
    # --- HEALTHY & NUTRITION ---
    "vihannes", "vihannekset", "kasvis", "kasvikset", "juures", "juurekset", "salaatti", 
    "hedelmä", "hedelmät", "marja", "marjat", "pähkinä", "pähkinät", "siemenet",
    "kuitu", "kuitua", "proteiini", "valkuainen", "vitamiini", "hivenaine",
    "terveellinen", "terveys", "ruokavalio", "vähärasvainen", "kevyt", "rasvaton",
    "kotiruoka", "itsetehty", "tuore", "lähiruoka", "peruna", "porkkana", "sipuli",
    "kala", "lohi", "kana", "broileri", "liha", "naudanliha", "kananmuna", "maito",

    # --- UNHEALTHY & CONVENIENCE ---
    "pikaruoka", "roskaruoka", "eines", "einekset", "valmisruoka", "sokeri", "sokeria",
    "makeinen", "karkki", "karkit", "suklaa", "herkku", "herkut", "rasvainen", 
    "epäterveellinen", "lihava", "lihavuus", "lihoaminen", "sipsit", "sipsi", "pizza", 
    "pitsa", "hampurilainen", "purilainen", "kebab", "ranskalaiset", "limsa", "limu", 
    "alkoholi", "olut", "viini", "viina", "siideri", "prosessoitu", "lisäaine",

    # --- SUSTAINABILITY & ETHICS ---
    "kasvissyöjä", "kasvisruoka", "vegaani", "vegaaninen", "veg", "kaurajuoma", "soija", 
    "tofu", "härkis", "nyhtökaura", "lihankulutus", "luomu", "ilmastoystävällinen", 
    "hiilijalanjälki", "ilmasto", "päästöt", "ympäristö", "kotimainen", "suomalainen", 
    "tuotanto", "hävikki", "ruokahävikki", "eettinen", "eläinsuojelu",

    # --- ECONOMY & SHOPPING ---
    "hinta", "halpa", "kallis", "tarjous", "ale", "euroa", "euro", "kauppa", "ruokakauppa",
    "market", "lidl", "prisma", "citymarket", "k-kauppa", "s-ryhmä", "budjetti", "opiskelija"
]

RELIGIOUS_BLACKLIST = [
    "jumala", "jumalan", "jumalasta", "jeesus", "jeesuksen", "kristus", "kristuksen", 
    "herra", "herran", "raamattu", "raamatun", "pyhä", "henki", "aamen", "evankeliumi", 
    "synti", "synnin", "armo", "pelastus", "seurakunta", "usko", "uskonto", "uskova", 
    "uskovat", "profetta", "opetuslapsi"
]

def build_temporal_thread_network(year):
    """Loads a yearly CSV, calculates thread similarity, and builds a network."""
    filename = f"suomi24_food_{year}.csv"
    
    if not os.path.exists(filename):
        print(f"File {filename} not found. Skipping...")
        return None
        
    print(f"\n--- Processing Year {year} ---")
    
    # 1. LOAD AND PREPARE DATA
    df = pd.read_csv(filename)
    df = df.dropna(subset=['post_content'])
    
    # First, group by thread so we have 'thread_df' defined
    thread_df = df.groupby('thread_id')['post_content'].apply(' '.join).reset_index()
    print(f"Total unique threads in {year}: {len(thread_df)}")

    # 2. RELIGIOUS FILTERING
    # Now we can safely use thread_df
    mask = thread_df['post_content'].str.contains('|'.join(RELIGIOUS_BLACKLIST), case=False, na=False)
    thread_df = thread_df[~mask].copy()
    print(f"Threads after religious filtering: {len(thread_df)}")

    # 3. SUBSTANCE FILTER
    def count_whitelist_terms(text):
        text = str(text).lower()
        return sum(1 for word in FOOD_WHITELIST if word in text)

    print("Filtering for threads with enough 'food substance'...")
    thread_df['word_count'] = thread_df['post_content'].apply(count_whitelist_terms)
    # Keeping your setting of 5 for a tight network
    thread_df = thread_df[thread_df['word_count'] >= 5].copy()
    
    print(f"Threads remaining after filtering: {len(thread_df)}")
    if len(thread_df) < 2: 
        print("Not enough threads left after filtering. Skipping.")
        return None

    # 4. TF-IDF VECTORIZATION
    vectorizer = TfidfVectorizer(vocabulary=FOOD_WHITELIST)
    tfidf_matrix = vectorizer.fit_transform(thread_df['post_content'])

    # 5. COSINE SIMILARITY
    print("Calculating similarity matrix...")
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    # 6. BUILD THE NETWORK
    G = nx.Graph()
    thread_ids = thread_df['thread_id'].tolist()
    G.add_nodes_from(thread_ids)
    
    rows, cols = np.where(sim_matrix >= SIMILARITY_THRESHOLD)
    edges = []
    for r, c in zip(rows, cols):
        if r < c: 
            weight = float(sim_matrix[r, c])
            edges.append((thread_ids[r], thread_ids[c], weight))
            
    G.add_weighted_edges_from(edges)
    G.remove_nodes_from(list(nx.isolates(G)))
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    density = nx.density(G) if num_nodes > 1 else 0
    
    print(f"Network built! Nodes: {num_nodes}, Edges: {num_edges}, Density: {density:.4f}")
    
    output_graph = f"thread_network_{year}.gexf"
    nx.write_gexf(G, output_graph)
    print(f"Saved graph to {output_graph}")
    
    return G

# ==========================================
# RUN THE LOOP
# ==========================================
networks = {}
for y in YEARS:
    # Testing with 2018 first
    if y == 2018: 
        networks[y] = build_temporal_thread_network(y)
    
print("\nProcess finished!")


--- Processing Year 2018 ---
Total unique threads in 2018: 43076
Threads after religious filtering: 33590
Filtering for threads with enough 'food substance'...
Threads remaining after filtering: 2722
Calculating similarity matrix...
Network built! Nodes: 1282, Edges: 11085, Density: 0.0135
Saved graph to thread_network_2018.gexf

Process finished!


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load your original text data and the Gephi export
print("Loading data...")
df_text = pd.read_csv("suomi24_food_2018.csv")
df_gephi = pd.read_csv("nodes2018_0.60.csv")

# 2. Merge them together
# We match the 'thread_id' from the text file to the 'Id' column from Gephi
df_merged = pd.merge(df_text, df_gephi, left_on='thread_id', right_on='Id')

# Drop empty posts just in case
df_merged = df_merged.dropna(subset=['post_content'])

# 3. Analyze the top communities
# Find the largest communities (e.g., the top 5 largest colors in your graph)
top_communities = df_merged['modularity_class'].value_counts().head(5).index

FOOD_WHITELIST = [
    # --- HEALTHY & NUTRITION ---
    "vihannes", "vihannekset", "kasvis", "kasvikset", "juures", "juurekset", "salaatti", 
    "hedelmä", "hedelmät", "marja", "marjat", "pähkinä", "pähkinät", "siemenet",
    "kuitu", "kuitua", "proteiini", "valkuainen", "vitamiini", "hivenaine",
    "terveellinen", "terveys", "ruokavalio", "vähärasvainen", "kevyt", "rasvaton",
    "kotiruoka", "itsetehty", "tuore", "lähiruoka", "peruna", "porkkana", "sipuli",
    "kala", "lohi", "kana", "broileri", "liha", "naudanliha", "kananmuna", "maito",

    # --- UNHEALTHY & CONVENIENCE ---
    "pikaruoka", "roskaruoka", "eines", "einekset", "valmisruoka", "sokeri", "sokeria",
    "makeinen", "karkki", "karkit", "suklaa", "herkku", "herkut", "rasvainen", 
    "epäterveellinen", "lihava", "lihavuus", "lihoaminen", "sipsit", "sipsi", "pizza", 
    "pitsa", "hampurilainen", "purilainen", "kebab", "ranskalaiset", "limsa", "limu", 
    "alkoholi", "olut", "viini", "viina", "siideri", "prosessoitu", "lisäaine",

    # --- SUSTAINABILITY & ETHICS ---
    "kasvissyöjä", "kasvisruoka", "vegaani", "vegaaninen", "veg", "kaurajuoma", "soija", 
    "tofu", "härkis", "nyhtökaura", "lihankulutus", "luomu", "ilmastoystävällinen", 
    "hiilijalanjälki", "ilmasto", "päästöt", "ympäristö", "kotimainen", "suomalainen", 
    "tuotanto", "hävikki", "ruokahävikki", "eettinen", "eläinsuojelu",

    # --- ECONOMY & SHOPPING (The "Glue" that connects topics) ---
    "hinta", "halpa", "kallis", "tarjous", "ale", "euroa", "euro", "kauppa", "ruokakauppa",
    "market", "lidl", "prisma", "citymarket", "k-kauppa", "s-ryhmä", "budjetti", "opiskelija"
]

FINNISH_GRAMMAR_STOPWORDS = [
    # Verbs (Auxiliary & Common)
    "olla", "on", "ole", "oli", "olen", "olet", "olemme", "olette", "ovat", "olisi", "olleet", "ollut",
    "voida", "voi", "voivat", "voisi", "pitää", "pitäisi", "tulla", "tulee", "tuli", "saada", "saa", "saavat",
    "tehdä", "tekee", "teki", "mennä", "menee", "meni", "sanoa", "sanoo", "sanoi",
    
    # Conjunctions & Particles
    "ja", "ei", "en", "et", "emme", "ette", "eivät", "että", "tai", "vai", "vaan", "mutta", 
    "kun", "jos", "vaikka", "kuin", "myös", "niin", "sekä", "siis", "eli", "kyllä", "ehkä",
    
    # Pronouns
    "se", "sen", "sitä", "ne", "niiden", "niitä", "tämä", "tämän", "tätä", "nämä", "näiden", "näitä", 
    "tuo", "tuon", "tuota", "nuo", "joka", "jonka", "jota", "jotka", "joiden", "joita", 
    "mikä", "minkä", "mitä", "mitkä", "kuka", "ketkä", "kenen", "ketä",
    "minä", "minun", "minua", "mä", "mun", "mua", "me", "meidän", "meitä", 
    "sinä", "sinun", "sinua", "sä", "sun", "sua", "te", "teidän", "teitä", 
    "hän", "hänen", "häntä", "he", "heidän", "heitä", "itse", "itsensä",
    
    # Adverbs & Fillers
    "niin", "siis", "tosi", "ihan", "vain", "vielä", "jo", "nyt", "sitten", "taas", "heti", 
    "aina", "joskus", "no", "vähän", "paljon", "muutama", "usein", "kanssa", "mukaan", 
    "takia", "vuoksi", "läpi", "asti", "täällä", "siellä", "missä", "sinne", "tänne",
    
    # Corporate/Infrastructure Noise (From your results)
    "oy", "suomen", "suomessa", "keski", "kunta", "kunnat", "euroa", "miljoonaa"
]

print("\n=== COMMUNITY PROFILES ===")

for community_num in top_communities:
    # Isolate all the text from this specific community
    community_text = df_merged[df_merged['modularity_class'] == community_num]['post_content']
    
    # Count how many posts are in this community
    size = len(community_text)
    
    # We use TF-IDF again to extract the defining keywords of this group
    vectorizer = TfidfVectorizer(
        max_features=12,
        stop_words=FINNISH_GRAMMAR_STOPWORDS,
        max_df=0.6)
    
    try:
        tfidf_matrix = vectorizer.fit_transform(community_text)
        # Get the top 10 defining words
        keywords = vectorizer.get_feature_names_out()
        
        print(f"\nCommunity {community_num} ({size} posts):")
        print(f"Top keywords: {', '.join(keywords)}")
        
    except ValueError:
        print(f"\nCommunity {community_num} ({size} posts): Not enough text to analyze.")

Loading data...

=== COMMUNITY PROFILES ===

Community 173 (3401 posts):
Top keywords: hän, hänen, jumalan, kautta, lain, lihan, minä, mutta, sillä, vaan

Community 179 (1821 posts):
Top keywords: alkoholi, alkoholia, alkoholin, eikä, en, mutta, sitten, vaan, vain, voi

Community 210 (875 posts):
Top keywords: en, itse, lihava, mies, mutta, nainen, sitten, vaan, vain, voi

Community 189 (808 posts):
Top keywords: en, mitä, mutta, nyt, paljon, saa, sitten, sokeria, vain, voi

Community 95 (714 posts):
Top keywords: euroa, jo, keski, mutta, nyt, oy, saa, sitten, suomen, voi
